# Week 5 — Diagnostic Framework: Predicting Quantum Kernel Utility
**Project**: Quantum Kernels on Foundation-Model Embeddings

**THE PIVOTAL ANALYSIS — this determines whether you have an AAAI paper.**

**Three diagnostics tested as predictors of the quantum-classical gap:**
1. **KTA** (kernel-target alignment) — how well does the kernel match the labels?
2. **Concentration** (effective rank of kernel matrix) — is the kernel collapsing?
3. **Geometric difference** (Huang et al. 2021) — can classical replicate quantum?

**Runtime:** ~10 min (all computation is on saved matrices, no kernel recomputation)

In [ ]:
!pip install scikit-learn numpy scipy matplotlib --quiet
import numpy as np, json, pickle, os, shutil, glob, warnings
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')
print("Loaded.")

---
## Step 0: Storage

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)

# outputs from THIS analysis go into a week5 subfolder
OUT_DIR = os.path.join(SAVE_DIR, 'week5')
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Reading kernel matrices from: {SAVE_DIR}")
print(f"Saving week5 outputs to:      {OUT_DIR}")

def _find(fname):
    # inputs are read from the MAIN folder (kernel matrices live there)
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname

def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        dst = os.path.join(OUT_DIR, os.path.basename(p))
        shutil.copy(p, dst)
        print(f"  saved -> {dst}")

print("Helpers ready.")

---
## Step 1: Define the three diagnostic measures

In [ ]:
def kernel_target_alignment(K, y):
    y = np.asarray(y); classes = np.unique(y)
    Y = np.zeros((len(y), len(y)))
    for c in classes:
        m = (y == c).astype(float); Y += np.outer(m, m)
    n = K.shape[0]; H = np.eye(n) - np.ones((n, n)) / n
    Kc = H @ K @ H; Yc = H @ Y @ H
    num = np.sum(Kc * Yc); den = np.sqrt(np.sum(Kc*Kc) * np.sum(Yc*Yc))
    return float(num / den) if den > 0 else 0.0

def kernel_concentration(K):
    eigs = np.linalg.eigvalsh(K)
    eigs = np.maximum(eigs, 0)
    eigs = eigs / (eigs.sum() + 1e-12)
    entropy = -np.sum(eigs * np.log(eigs + 1e-12))
    eff_rank = np.exp(entropy) / K.shape[0]
    return float(eff_rank)

def geometric_difference(K_quantum, K_classical):
    # spectral proxy: ratio of effective dimensions (kept for comparison)
    eff_q = (np.trace(K_quantum)**2) / (np.sum(K_quantum**2) + 1e-10)
    eff_c = (np.trace(K_classical)**2) / (np.sum(K_classical**2) + 1e-10)
    return float(eff_q / (eff_c + 1e-10))

def geometric_difference_exact(K_Q, K_C, lam=1e-6):
    # Huang et al. 2021: sqrt of spectral norm of sqrt(K_C) K_Q^{-1} sqrt(K_C)
    # ... equivalently with roles as defined in their Eq. (5); we use the
    # standard form g = sqrt(|| sqrt(K_Q) K_C^{-1} sqrt(K_Q) ||_inf) with
    # trace normalization and ridge-regularized inverse for stability.
    n = K_Q.shape[0]
    K_Q = K_Q * n / np.trace(K_Q)
    K_C = K_C * n / np.trace(K_C)
    w, V = np.linalg.eigh(K_Q)
    w = np.maximum(w, 0)
    sqrtKQ = (V * np.sqrt(w)) @ V.T
    Kc_inv = np.linalg.inv(K_C + lam * n * np.eye(n))
    M = sqrtKQ @ Kc_inv @ sqrtKQ
    return float(np.sqrt(max(np.linalg.eigvalsh(M).max(), 0.0)))

print("Three diagnostics defined: KTA, concentration, geometric difference.")

---
## Step 2: Scan Drive folder and build configuration registry

In [ ]:
all_files = sorted(os.listdir(SAVE_DIR))
kernel_files = [f for f in all_files if f.endswith('_kernel_matrices.npz')]
print(f"Found {len(kernel_files)} standard kernel matrix files:")
for f in kernel_files: print(f"  {f}")

configs = []

# --- LEGACY: original SST-2 BERT files use non-standard names ---
# kernel_matrices_week3.npz + embeddings_week2.npz (the very first dataset)
legacy_km = _find('kernel_matrices_week3.npz')
legacy_emb = _find('embeddings_week2.npz')
legacy_bl = _find('baseline_results.pkl')
if os.path.exists(legacy_km) and os.path.exists(legacy_emb):
    # baseline file might be 'baseline_results.pkl' or 'results_week3.json'-derived
    bl_ok = os.path.exists(legacy_bl)
    if not bl_ok:
        # try alternative legacy baseline names
        for alt in ['baseline_results.pkl', 'sst2_baseline_results.pkl', 'results_week2.pkl']:
            if os.path.exists(_find(alt)):
                legacy_bl = _find(alt); bl_ok = True; break
    if bl_ok:
        configs.append({'tag': 'sst2_bert', 'kernel_file': legacy_km,
                        'emb_file': legacy_emb, 'bl_file': legacy_bl, 'metric': 'binary'})
        print("  [legacy] added sst2_bert (kernel_matrices_week3.npz + embeddings_week2.npz)")
    else:
        print("  [legacy] SST-2 BERT kernel+emb found but no baseline pkl — skipping")
        print("           (if you have SST-2 BERT baselines under another name, tell Claude)")

# --- STANDARD: all {tag}_kernel_matrices.npz files ---
for kf in kernel_files:
    tag = kf.replace('_kernel_matrices.npz', '')
    emb_file = f"{tag}_embeddings.npz"
    bl_file = f"{tag}_baseline_results.pkl"
    emb_path = _find(emb_file); bl_path = _find(bl_file); km_path = _find(kf)
    if not os.path.exists(emb_path):
        print(f"  [skip] {tag}: no embeddings"); continue
    if not os.path.exists(bl_path):
        print(f"  [skip] {tag}: no baselines"); continue
    if any(x in tag for x in ['agnews', 'ag_news', 'snlive', 'multiling']):
        metric = 'macro'
    else:
        metric = 'binary'
    configs.append({'tag': tag, 'kernel_file': km_path, 'emb_file': emb_path,
                    'bl_file': bl_path, 'metric': metric})

print(f"\n{len(configs)} valid configurations found for analysis.")
print("Tags:", [c['tag'] for c in configs])

---
## Step 3: Compute all three diagnostics + quantum-classical gap

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

results = []
print(f"{'Config':<35} {'Q-F1':>6} {'C-F1':>6} {'Gap':>7} {'KTA':>7} {'Conc':>7} {'GeoDiff':>8} {'gExact':>8}")
print("-"*82)

for cfg in configs:
    tag = cfg['tag']
    try:
        km = np.load(cfg['kernel_file'])
        K_q_train = km['K_train_quantum']
        emb = np.load(cfg['emb_file'])
        X_tr = emb['X_tr_subset'].astype(np.float64)
        y_tr = emb['y_tr_subset']
        X_te = emb['X_te_subset'].astype(np.float64)
        y_te = emb['y_te_subset']
        with open(cfg['bl_file'], 'rb') as f: baselines = pickle.load(f)
        avg = cfg['metric']
        km_test = km['K_test_quantum']
        yp = SVC(kernel='precomputed', C=1.0).fit(K_q_train, y_tr).predict(km_test)
        q_f1 = float(f1_score(y_te, yp, average=avg))
        best_c_f1 = 0
        for name, v in baselines.items():
            f = v.get('f1', v.get('f1_macro', 0)); best_c_f1 = max(best_c_f1, f)
        gap = q_f1 - best_c_f1
        kta = kernel_target_alignment(K_q_train, y_tr)
        conc = kernel_concentration(K_q_train)
        K_rbf = rbf_kernel(X_tr, X_tr, gamma=1.0/X_tr.shape[1])
        geo = geometric_difference(K_q_train, K_rbf)
        geo_exact = geometric_difference_exact(K_q_train, K_rbf)
        results.append({'tag': tag, 'quantum_f1': q_f1, 'best_classical_f1': best_c_f1,
            'gap': gap, 'kta': kta, 'concentration': conc, 'geometric_difference': geo, 'geometric_difference_exact': geo_exact, 'metric': avg})
        print(f"{tag:<35} {q_f1:>6.3f} {best_c_f1:>6.3f} {gap:>+7.3f} {kta:>7.4f} {conc:>7.4f} {geo:>8.4f} {geo_exact:>8.2f}")
    except Exception as e:
        print(f"{tag:<35} ERROR: {e}")

print(f"\nSuccessfully analyzed {len(results)} configurations.")

---
## Step 4: Correlation — do the diagnostics predict the gap?

In [ ]:
if len(results) < 3:
    print("Need at least 3 configs for correlation.")
else:
    gaps = np.array([r['gap'] for r in results])
    ktas = np.array([r['kta'] for r in results])
    concs = np.array([r['concentration'] for r in results])
    geos = np.array([r['geometric_difference'] for r in results])
    geos_exact = np.array([r['geometric_difference_exact'] for r in results])
    tags = [r['tag'] for r in results]

    diagnostics = [('KTA', ktas), ('Concentration', concs), ('Geometric Difference', geos_exact)]
    print("="*60)
    print("CORRELATION: diagnostic vs quantum-classical gap")
    print("="*60)
    corr_results = {}
    for name, vals in diagnostics:
        pr, pp = pearsonr(vals, gaps); sr, sp = spearmanr(vals, gaps)
        corr_results[name] = {'pearson_r': float(pr), 'pearson_p': float(pp),
                              'spearman_r': float(sr), 'spearman_p': float(sp)}
        sig = "***" if pp<0.01 else ("**" if pp<0.05 else ("*" if pp<0.1 else ""))
        print(f"  {name:<25} Pearson r={pr:+.3f} (p={pp:.3f}){sig}  Spearman r={sr:+.3f} (p={sp:.3f})")
    print("\nSignificance: *** p<0.01, ** p<0.05, * p<0.1")
    best_diag = max(corr_results, key=lambda k: abs(corr_results[k]['pearson_r']))
    best_r = corr_results[best_diag]['pearson_r']
    pr_px, pp_px = pearsonr(geos, gaps)
    print(f"  {'GeoDiff proxy (eff-dim ratio)':<25} Pearson r={pr_px:+.3f} (p={pp_px:.3f})   [appendix]")
    print(f"\nBest single predictor: {best_diag} (|r| = {abs(best_r):.3f})")

---
## Step 5: Scatter plots — the key figure

In [ ]:
if len(results) >= 3:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    for ax, (name, vals) in zip(axes, diagnostics):
        ax.scatter(vals, gaps, s=50, color='#D95F02', edgecolor='white', alpha=0.8, zorder=3)
        if len(vals) >= 2 and np.std(vals) > 1e-10:
            z = np.polyfit(vals, gaps, 1); xs = np.linspace(vals.min(), vals.max(), 50)
            ax.plot(xs, np.poly1d(z)(xs), '--', color='gray', alpha=0.7)
        mean_gap = np.mean(gaps); std_gap = np.std(gaps) + 1e-10
        for t, xv, yv in zip(tags, vals, gaps):
            if abs(yv - mean_gap) > 0.8*std_gap or len(results) <= 15:
                label = t.replace('multiling_','ml_').replace('_siglip','_sig')[:18]
                ax.annotate(label, (xv,yv), fontsize=6, alpha=0.7,
                           textcoords="offset points", xytext=(4,3))
        pr = corr_results[name]['pearson_r']; pp = corr_results[name]['pearson_p']
        ax.set_xlabel(name, fontsize=12); ax.set_ylabel('Quantum - Classical F1 gap', fontsize=11)
        ax.set_title(f'{name}\nPearson r={pr:+.3f} (p={pp:.3f})', fontsize=11)
        ax.axhline(0, color='black', lw=0.8, alpha=0.4); ax.grid(True, alpha=0.2)
    plt.suptitle(f'Diagnostic Framework: predicting quantum kernel utility\n'
                 f'across {len(results)} configurations', fontsize=13, y=1.04)
    plt.tight_layout(); plt.show()
    save_figure(fig, 'week5_diagnostic_scatter_3panel')

---
## Step 6: Combined multi-diagnostic predictor

In [ ]:
if len(results) >= 5:
    from sklearn.linear_model import LinearRegression
    from sklearn.model_selection import LeaveOneOut
    from sklearn.metrics import r2_score
    X_diag = np.column_stack([ktas, concs, geos_exact]); y_gap = gaps
    lr = LinearRegression().fit(X_diag, y_gap)
    r2_full = r2_score(y_gap, lr.predict(X_diag))
    print(f"Combined 3-diagnostic linear model:")
    print(f"  R2 (full data) = {r2_full:.3f}")
    print(f"  Coefficients: KTA={lr.coef_[0]:+.4f}, Conc={lr.coef_[1]:+.4f}, GeoDiff={lr.coef_[2]:+.4f}")
    loo = LeaveOneOut(); preds_loo = np.zeros(len(y_gap))
    for train_ix, test_ix in loo.split(X_diag):
        lr_loo = LinearRegression().fit(X_diag[train_ix], y_gap[train_ix])
        preds_loo[test_ix] = lr_loo.predict(X_diag[test_ix])
    r2_loo = r2_score(y_gap, preds_loo)
    print(f"  R2 (leave-one-out CV) = {r2_loo:.3f}")
    fig2, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(preds_loo, y_gap, s=60, color='#D95F02', edgecolor='white', zorder=3)
    mn, mx = min(preds_loo.min(), y_gap.min()), max(preds_loo.max(), y_gap.max())
    ax.plot([mn,mx],[mn,mx], 'k--', alpha=0.5, label='perfect prediction')
    ax.set_xlabel('Predicted gap (LOO-CV)', fontsize=12)
    ax.set_ylabel('Actual quantum - classical gap', fontsize=12)
    ax.set_title(f'Combined diagnostic predictor (LOO-CV R2 = {r2_loo:.3f})', fontsize=12)
    ax.legend(); ax.grid(True, alpha=0.2); plt.tight_layout(); plt.show()
    save_figure(fig2, 'week5_combined_predictor')
else:
    r2_loo = None
    print("Need >=5 configs for combined model. You have", len(results))

---
## Step 7: Summary + save

In [ ]:
print("\n" + "="*90)
print("FULL DIAGNOSTIC TABLE")
print("="*90)
print(f"{'Config':<30} {'Q-F1':>6} {'C-F1':>6} {'Gap':>7} {'KTA':>7} {'Conc':>7} {'GeoDiff':>8}")
print("-"*90)
for r in sorted(results, key=lambda x: x['gap']):
    print(f"{r['tag']:<30} {r['quantum_f1']:>6.3f} {r['best_classical_f1']:>6.3f} "
          f"{r['gap']:>+7.3f} {r['kta']:>7.4f} {r['concentration']:>7.4f} {r['geometric_difference']:>8.4f}")
output = {'n_configs': len(results), 'results': results,
          'correlations': corr_results if 'corr_results' in dir() else {},
          'best_predictor': best_diag if 'best_diag' in dir() else None,
          'combined_r2_loo': float(r2_loo) if r2_loo is not None else None}
with open(os.path.join(OUT_DIR, 'week5_diagnostic_analysis.json'), 'w') as f:
    json.dump(output, f, indent=2)
# regularization sensitivity for exact g (preempts the lambda objection)
print("\nExact-g regularization sweep (Pearson r vs gap):")
lam_sweep = {}
for lam in [1e-4, 1e-5, 1e-6, 1e-7]:
    gvals = []
    for cfg2 in configs:
        try:
            km2 = np.load(cfg2['kernel_file']); Kq2 = km2['K_train_quantum']
            X2 = np.load(cfg2['emb_file'])['X_tr_subset'].astype(np.float64)
            Kr2 = rbf_kernel(X2, X2, gamma=1.0/X2.shape[1])
            gvals.append(geometric_difference_exact(Kq2, Kr2, lam=lam))
        except Exception:
            gvals.append(np.nan)
    gv = np.array(gvals); ok = ~np.isnan(gv)
    rr = pearsonr(gv[ok], np.array([r['gap'] for r in results])[ok])[0] if ok.sum()>2 else float('nan')
    lam_sweep[str(lam)] = float(rr)
    print(f"  lam={lam:.0e}: r={rr:+.3f}")
output['exact_g_lambda_sweep'] = lam_sweep
with open(os.path.join(OUT_DIR, 'week5_diagnostic_analysis.json'), 'w') as f:
    json.dump(output, f, indent=2)
print(f"\nSaved: week5_diagnostic_analysis.json ({len(results)} configs)")

---
## Step 8: The verdict

In [ ]:
if 'best_diag' in dir():
    br = abs(corr_results[best_diag]['pearson_r'])
    r2 = r2_loo if r2_loo is not None else 0
    print("="*60)
    print("VERDICT")
    print("="*60)
    if br > 0.5 or r2 > 0.3:
        print("POSITIVE: The diagnostics show predictive power.")
        print("  -> Reframe as diagnostic framework for quantum kernel utility")
        print("  -> AAAI 2027 is a genuine shot (July 28 deadline)")
        print("  -> The experimental grid is the validation set, not the contribution")
    elif br > 0.3:
        print("MODERATE: Some predictive signal but not overwhelming.")
        print("  -> Report honestly, frame as preliminary evidence")
        print("  -> Target: Quantum Machine Intelligence (Q1) or ICLR 2027")
        print("  -> Consider adding configs or combining diagnostics")
    else:
        print("WEAK/NONE: Diagnostics do not reliably predict the gap.")
        print("  -> Clean negative result, publishable and valuable")
        print("  -> Target: TMLR or Quantum Machine Intelligence (Q1)")
        print("  -> Frame: established diagnostics fail in the frozen-embedding regime")
        print("  -> A thorough negative from a careful study gets cited")
    print("="*60)

---
## Week 5 complete

### Saved:
- `week5_diagnostic_scatter_3panel.png/.pdf`
- `week5_combined_predictor.png/.pdf`
- `week5_diagnostic_analysis.json`

### Bring to your professor:
The correlation numbers and the scatter plot. They decide the venue.